# Databricks FinOps Starter — Regional Ingestion Pipeline

Thin entry point. **All** business logic lives in `src/finops/` (Python) and `src/sql/`
(SQL files); this notebook only wires parameters and calls `run_pipeline`.

Lifecycle per source: discover → validate contracts → incremental read (watermark
+ lookback) → regional Bronze MERGE → reconciliation → Silver → data quality →
checkpoint commit → (publisher → Gold). Failures are isolated per source and
recorded in `ops.*`.

In [0]:
dbutils.widgets.text("environment", "dev", "Environment (dev|test|prod)")
dbutils.widgets.text("sources", "", "Optional comma-separated source ids (empty = all enabled)")

ENVIRONMENT = dbutils.widgets.get("environment")
SOURCES = [s.strip() for s in dbutils.widgets.get("sources").split(",") if s.strip()] or None

In [0]:
import sys
from pathlib import Path

REPO = Path("/Workspace/Users/dhirajsardal@gmail.com/databricks-finops-starter")
sys.path.insert(0, str(REPO / "src"))

from finops.pipeline import run_pipeline  # noqa: E402

report = run_pipeline(
    spark,
    conf_root=REPO / "conf",
    environment=ENVIRONMENT,
    sources=SOURCES,
)
display(report)